## Парсинг данных с сайта на Python

### Определения

Парсинг (англ. parsing — разбор) — это процесс автоматического анализа веб-сайтов для сбора структурированной информации. Еще парсинг часто называют веб-скрапингом. 

Парсинг - это процесс сбора данных с последующей их обработкой и анализом.

Программа, которая занимается парсингом, называют - парсер.


### Условие задачи

С сайта ( https://habr.com/ru/search/ ) необходимо построить исходный набор данных (.csv или .xml). Набор данных должен включать __название, описание, рейтинг и сферу деятельности компаний, дату публикации, а также текст статей из Интернет-ресурсов__. Подготовленный набор данных должен содержать сведения о всех номинантах конкурса. Разработанный парсер должен извлекать гиперссылки из начальной страницы с последующим обходом всех страниц по полученным ссылкам и извлечением их содержимого. Можно дополнить набор какими-либо другими данными, если они могут быть полезны для дальнейшего исследования.


### Этапы парсинга

1. Поиск данных
2. Получение информации
3. Сохранение данных

### Подключение библиотек

In [1]:
from bs4 import BeautifulSoup as bs

Beautiful Soup - это библиотека Python для извлечения данных из HTML и XML файлов. 

In [2]:
import requests

Библиотека requests является стандартным инструментом для составления HTTP-запросов в Python.

In [3]:
import pandas as pd

### Получение информаций

In [4]:

url = 'https://habr.com/ru/all/' 
page = requests.get(url)

Метод __requests.get(url)__ из библиотеки requests в Python выполняет HTTP-запрос типа GET по указанному URL. Этот запрос используется для получения данных с веб-страницы или API, в нашем случае из страницы habr.

In [5]:
page.status_code

200

Если вызвать __page.status_code__, то получим статус состояния HTTP. например, 200 — успешно, 404 — страница не найдена, 500 — ошибка сервера 


In [6]:
soup = bs(page.text, 'html.parser')

__bs__ — это сокращение от BeautifulSoup, основного класса библиотеки Beautiful Soup.

__bs(page.text, 'html.parser')__ создаёт объект BeautifulSoup, который парсит HTML-код из page.text с использованием указанного парсера.

__'html.parser'__ — это встроенный парсер Python, который не требует установки дополнительных библиотек.а.

In [7]:
page.text

'<!DOCTYPE html>\n<html lang="ru">\n\n  <head>\n    <title>Все статьи подряд &#x2F; Хабр</title>\n<link rel="image_src" href="/img/habr_ru.png" data-hid="2a79c45">\n<link href="https://habr.com/ru/articles/" rel="canonical" data-hid="e3fa780">\n<link href="https://habr.com/ru/articles/" hreflang="ru" rel="alternate" data-hid="7d51b8a">\n<link href="https://habr.com/en/articles/" hreflang="en" rel="alternate" data-hid="7d51b8a">\n<meta itemprop="image" content="/img/habr_ru.png">\n<meta property="og:image" content="/img/habr_ru.png">\n<meta property="og:image:width" content="1200">\n<meta property="og:image:height" content="630">\n<meta property="aiturec:image" content="/img/habr_ru.png">\n<meta name="twitter:image" content="/img/habr_ru.png">\n<meta property="vk:image" content="/img/habr_ru.png?format=vk">\n<meta property="fb:app_id" content="444736788986613">\n<meta property="fb:pages" content="472597926099084">\n<meta name="twitter:card" content="summary_large_image">\n<meta name="tw

In [8]:
soup

<!DOCTYPE html>

<html lang="ru">
<head>
<title>Все статьи подряд / Хабр</title>
<link data-hid="2a79c45" href="/img/habr_ru.png" rel="image_src"/>
<link data-hid="e3fa780" href="https://habr.com/ru/articles/" rel="canonical"/>
<link data-hid="7d51b8a" href="https://habr.com/ru/articles/" hreflang="ru" rel="alternate"/>
<link data-hid="7d51b8a" href="https://habr.com/en/articles/" hreflang="en" rel="alternate"/>
<meta content="/img/habr_ru.png" itemprop="image"/>
<meta content="/img/habr_ru.png" property="og:image"/>
<meta content="1200" property="og:image:width"/>
<meta content="630" property="og:image:height"/>
<meta content="/img/habr_ru.png" property="aiturec:image"/>
<meta content="/img/habr_ru.png" name="twitter:image"/>
<meta content="/img/habr_ru.png?format=vk" property="vk:image"/>
<meta content="444736788986613" property="fb:app_id"/>
<meta content="472597926099084" property="fb:pages"/>
<meta content="summary_large_image" name="twitter:card"/>
<meta content="@habr_com" name=

Создадим словарь, в который будем записывать данные по заданию: название статьи, описание, рейтинг и сферу деятельности компаний, дату публикации, а также текст статьи из Интернет-ресурса

In [49]:
result_list = {'title': [], 'namecompany': [], 'description': [], 'rating': [], 'field': [], 'date': [], 'textpub': []}

### Алгоритм

Суть алгоритма заключается в переборе страниц, и переходе на "вложенные" страницы, то есть у нас есть основная страница https://habr.com/ru/all/, мы перебираем несколько стараниц с page1 до page10. На каждой странице есть статьи, записываем их в список, чтобы перейти по ним используем  _-i.a.get('href')-_  то есть берём значение из href этого заголовка. Далее находим классы элементов которые нам нужны, и записываем их в результат.

In [50]:
pagenum = 1
for i in range(10):
    url = 'https://habr.com/ru/feed/page' + str(pagenum) + '/' # переход на ссылуку с определённым номером сраницы
    page = requests.get(url)
    soup = bs(page.text, 'html.parser')
    titles = soup.find_all('h2', class_='tm-title tm-title_h2')# получаем заголовки всех статей на этой странице
    
    for i in titles: 
        # переход на страницу статьи
        url = 'https://habr.com' + str(i.a.get('href')) 
        page = requests.get(url)
        soup = bs(page.text, 'html.parser')
        
        name_company = soup.find('a', class_='tm-company-snippet__title')# получаем название компаний
        desc_company = soup.find('div', class_='tm-company-snippet__description')# получаем описание компаний
        
        if (name_company is not None): #если на странице присутсвует компания
        
            result_list['title'].append(i.text) # записываем название статьи
            result_list['namecompany'].append(name_company.text) # записываем название компании
            result_list['description'].append(desc_company.text) # записываем описание компании
            
            datepub = soup.find('span', class_='tm-article-datetime-published') # находим дату публикаций
            result_list['date'].append(datepub.time['datetime'][0: 10]) # записываем дату публикаций
            
            # текст статьи
            try:
                textpub = soup.find('div', class_='article-formatted-body article-formatted-body article-formatted-body_version-2').get_text()
                textpub = textpub.replace('\n', ' ').replace('\t', ' ').replace('\xa0', ' ').replace('\u200e', ' ').replace('\r', ' ')
            except:
                textpub = soup.find('div', class_='article-formatted-body article-formatted-body article-formatted-body_version-1').get_text()
                textpub = textpub.replace('\n', ' ').replace('\t', ' ').replace('\xa0', ' ').replace('\u200e', ' ').replace('\r', ' ')
            result_list['textpub'].append(textpub)
            
            # переход на страницу компании
            url = 'https://habr.com' + str(name_company.get('href'))
            page = requests.get(url)
            soup = bs(page.text, 'html.parser')
            
            #записываем рейтинг
            rating = soup.find('span', class_='tm-votes-lever__score-counter tm-votes-lever__score-counter_rating tm-votes-lever__score-counter')
            if(rating is None):
                result_list['rating'].append('0')
            else:
                result_list['rating'].append((rating.text).strip())
               
             #записываем отрасли компаний
            fieldtext = ""
            fields = soup.find_all('a', 'tm-company-profile__categories-text')
            for field in fields:
                fieldtext = fieldtext + ((field.text).strip()) + ", "
            if (fields is None):
                result_list['field'].append(None)
            else:
                result_list['field'].append(fieldtext[0:-2])
            
    pagenum += 1

Что бы найти элемет на странице, выделите этот элемент(заголовок, текст, изображение) и нажмите Ctrl + Shift + I или ПКМ и "Исследовать элемент".

In [51]:
result_list

{'title': ['Как мы прокачиваем HealthScore для 6000+ витрин и готовим DWH к AI',
  'Поднимаем интернет-радио на AzuraCast',
  'Мир ИТ и открытого ПО в 2026-м: в бой идут одни мейнтейнеры',
  'В поиске секретов. iOS Пентест. Часть третья',
  'Принципы организационного проектирования',
  'Атаки на цепочку поставки ПО: виды угроз и как с ними бороться',
  'Сравнение нейросетей в генерациях: Claude vs ChatGPT vs Gemini в различных сферах',
  'Как заставить Android Studio ругаться на код по вашим правилам: создаем пользовательские Lint и Detekt',
  'Genshin Impact: как купить кристаллы, Лун�� и боевой пропуск в России в 2026 году',
  'Mistral AI показала Leanstral: кодинг, который можно не проверять',
  'Новый ЦОД Рег.облака в Москве и зачем там GPU',
  'От стратегии до проектов: для чего бизнесу переходить к портфельному управлению',
  'Кейс компании ДАЛЕЕ: как они закрыли вакансию Специалист технической поддержки L1, Junior на Хабр Карьере',
  'Что случилось с CGI, и как FastCGI спас веб 

In [52]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
namecompany - 0
description - 0
rating - 0
field - 0
date - 0
textpub - 0


### Сохранение данных

In [53]:
file_name = 'habr.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)

In [54]:
df.head(15)

,title,namecompany,description,rating,field,date,textpub
0,Как мы прокачиваем HealthScore для 6000+ витри...,AvitoTech,У нас живут ваши объявления,437.8,"Веб-разработка, Электронная коммерция, Веб-сер...",2026-03-17,"Привет! Меня зовут Дмитрий Мележиков, я отвеча..."
1,Поднимаем интернет-радио на AzuraCast,HOSTKEY,"IT-инфраструктура: сервера, VPS, GPU, коло",77.79,"Аппаратное обеспечение, Связь и телекоммуникац...",2026-03-17,"Автор: Иван Богданов, Технический писатель Рад..."
2,Мир ИТ и открытого ПО в 2026-м: в бой идут одн...,МТС,Про жизнь и развитие в IT,1805.66,"Связь и телекоммуникации, Мобильные технологии...",2026-03-17,"Начало 2026 года. Скотт Шембо, мейнтейнер Matp..."
3,В поиске секретов. iOS Пентест. Часть третья,Технократия,Компания,19.28,"Веб-разработка, Веб-сервисы, Информационная бе...",2026-03-17,Вас приветствует Ян - старший пентестер из ком...
4,Принципы организационного проектирования,OTUS,"Развиваем технологии, обучая их создателей",751.28,"Консалтинг и поддержка, Рекрутинг и HR, Произв...",2026-03-17,"«А вы, друзья, как ни садитесь, всё в музыкант..."
5,Атаки на цепочку поставки ПО: виды угроз и как...,CodeScoring,Инструменты безопасной и качественной разработки,7.2,"Программное обеспечение, Информационная безопа...",2026-03-17,Атаки на цепочку поставки – одна из самых усто...
6,Сравнение нейросетей в генерациях: Claude vs C...,BotHub,Российский Openrouter,434.95,"Веб-разработка, Программное обеспечение, Веб-с...",2026-03-17,Новые модели выходят практически каждый день. ...
7,Как заставить Android Studio ругаться на код п...,Совкомбанк Технологии,Меняй мир финансовых технологий вместе с нами,72.85,"Веб-разработка, Программное обеспечение, Мобил...",2026-03-17,"Привет, Хабр! На связи Алина, старший Android-..."
8,"Genshin Impact: как купить кристаллы, Лун�� и ...",ggsel.net,Компания,84.4,"Программное обеспечение, Игры и развлечения",2026-03-17,Пополнить баланс примогемов или купить Луну иг...
9,"Mistral AI показала Leanstral: кодинг, который...",BotHub,Российский Openrouter,434.95,"Веб-разработка, Программное обеспечение, Веб-с...",2026-03-17,Французская Mistral AI представила Leanstral –...


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        78 non-null     object
 1   namecompany  78 non-null     object
 2   description  78 non-null     object
 3   rating       78 non-null     object
 4   field        78 non-null     object
 5   date         78 non-null     object
 6   textpub      78 non-null     object
dtypes: object(7)
memory usage: 4.4+ KB


### Дополнительные 20 статей

взяты с сайта недельных статей

In [56]:
existing_titles =[]
for i in range(len(df)):
    existing_titles.append(df["title"] + " "+ df["namecompany"])

In [57]:
existing_titles

[0     Как мы прокачиваем HealthScore для 6000+ витри...
 1         Поднимаем интернет-радио на AzuraCast HOSTKEY
 2     Мир ИТ и открытого ПО в 2026-м: в бой идут одн...
 3     В поиске секретов. iOS Пентест. Часть третья Т...
 4         Принципы организационного проектирования OTUS
                             ...                        
 73          Почему пользователи ненавидят PAM? Avanpost
 74    Security Week 2612: ненастоящая уязвимость в а...
 75    Uber и Nvidia планируют запуск роботакси в 28 ...
 76    GPT-4.5 смогла обмануть 73% людей, лишь притво...
 77    “Encyclopedia Britannica” подаёт в суд на Open...
 Length: 78, dtype: object,
 0     Как мы прокачиваем HealthScore для 6000+ витри...
 1         Поднимаем интернет-радио на AzuraCast HOSTKEY
 2     Мир ИТ и открытого ПО в 2026-м: в бой идут одн...
 3     В поиске секретов. iOS Пентест. Часть третья Т...
 4         Принципы организационного проектирования OTUS
                             ...                        
 73

In [58]:
existing_titles_set = set()  
count = 0
pagenum = 1
for page_num in range(10): 
    if count >= 20:
        break
    url = f'https://habr.com/ru/articles/top/weekly/page{pagenum}/'
    
    
    page = requests.get(url)
    soup = bs(page.text, 'html.parser')
    
    titles = soup.find_all('h2', class_='tm-title tm-title_h2')
    
    for title in titles: 
        if count >= 20:
            break
            
        article_url = 'https://habr.com' + str(title.a.get('href'))
        article_page = requests.get(article_url)
        article_soup = bs(article_page.text, 'html.parser')
        
        name_company = article_soup.find('a', class_='tm-company-snippet__title')
        desc_company = article_soup.find('div', class_='tm-company-snippet__description')
        
        if name_company is not None:  
            combined_title = f"{title.text} {name_company.text}"
            
            company_url = 'https://habr.com' + str(name_company.get('href'))
            company_page = requests.get(company_url)
            company_soup = bs(company_page.text, 'html.parser')
            
            rating = company_soup.find('span', class_='tm-votes-lever__score-counter tm-votes-lever__score-counter_rating tm-votes-lever__score-counter')
            
            fieldtext = ""
            fields = company_soup.find_all('a', 'tm-company-profile__categories-text')
            for field in fields:
                fieldtext += field.text.strip() + ", "
            
            has_complete_info = (
                desc_company is not None and 
                rating is not None and 
                len(fields) > 0
            )
            
            if has_complete_info and combined_title not in existing_titles_set:
                existing_titles_set.add(combined_title)
                
                result_list['title'].append(title.text)
                result_list['namecompany'].append(name_company.text)
                result_list['description'].append(desc_company.text)
                
                datepub = article_soup.find('span', class_='tm-article-datetime-published')
                if datepub and datepub.time and datepub.time.get('datetime'):
                    result_list['date'].append(datepub.time['datetime'][0:10])
                else:
                    result_list['date'].append(None)
                
                try:
                    textpub = article_soup.find('div', class_='article-formatted-body article-formatted-body article-formatted-body_version-2')
                    if not textpub:
                        textpub = article_soup.find('div', class_='article-formatted-body article-formatted-body article-formatted-body_version-1')
                    
                    if textpub:
                        textpub = textpub.get_text()
                        textpub = textpub.replace('\n', ' ').replace('\t', ' ').replace('\xa0', ' ').replace('\u200e', ' ').replace('\r', ' ')
                    else:
                        textpub = ""
                except:
                    textpub = ""
                result_list['textpub'].append(textpub)
                
                result_list['rating'].append(rating.text.strip())
                result_list['field'].append(fieldtext[:-2])
                
                count += 1
                print(f"Добавлена статья {count}: {title.text[:50]}...")
    
    pagenum += 1

df2 = pd.DataFrame(result_list)
print(f"DataFrame shape: {df2.shape}")
print(f"Уникальных статей добавлено: {count}")
print(f"Количество уникальных названий в set: {len(existing_titles_set)}")
print(df2.head())

Добавлена статья 1: Мой плохой код — это ваша вина...
Добавлена статья 2: ИИ-дед возвращается. Реализация ИИ-собеседника для...
Добавлена статья 3: Считаем логарифмы в уме...
Добавлена статья 4: Никого не повышают за простые решения...
Добавлена статья 5: После блокировки мессенджеров Matrix стал единстве...
Добавлена статья 6: Как мы отправили сисадмина поставить сервер в Анта...
Добавлена статья 7: Анализ плутония из советского датчика дыма...
Добавлена статья 8: «О! Айтишники! Вы же полезными стали. А можете нам...
Добавлена статья 9: Почему не взлетели дирижабли? Часть 13: сквозь Сев...
Добавлена статья 10: ArcaOS 5.1.2: как OS/2 добралась до UEFI и больших...
Добавлена статья 11: Как Нейроюрист ищет по миллионам юридических докум...
Добавлена статья 12: Как заставить англоязычную нейросеть читать по-рус...
Добавлена статья 13: Похмелье после электроэйфории: почему вокруг элект...
Добавлена статья 14: Walking-working: как я создал собственный сетап дл...
Добавлена статья 15: Хьюсто

In [59]:
result_list

{'title': ['Как мы прокачиваем HealthScore для 6000+ витрин и готовим DWH к AI',
  'Поднимаем интернет-радио на AzuraCast',
  'Мир ИТ и открытого ПО в 2026-м: в бой идут одни мейнтейнеры',
  'В поиске секретов. iOS Пентест. Часть третья',
  'Принципы организационного проектирования',
  'Атаки на цепочку поставки ПО: виды угроз и как с ними бороться',
  'Сравнение нейросетей в генерациях: Claude vs ChatGPT vs Gemini в различных сферах',
  'Как заставить Android Studio ругаться на код по вашим правилам: создаем пользовательские Lint и Detekt',
  'Genshin Impact: как купить кристаллы, Лун�� и боевой пропуск в России в 2026 году',
  'Mistral AI показала Leanstral: кодинг, который можно не проверять',
  'Новый ЦОД Рег.облака в Москве и зачем там GPU',
  'От стратегии до проектов: для чего бизнесу переходить к портфельному управлению',
  'Кейс компании ДАЛЕЕ: как они закрыли вакансию Специалист технической поддержки L1, Junior на Хабр Карьере',
  'Что случилось с CGI, и как FastCGI спас веб 

In [60]:

df2 = pd.DataFrame(data=result_list)


In [61]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        98 non-null     object
 1   namecompany  98 non-null     object
 2   description  98 non-null     object
 3   rating       98 non-null     object
 4   field        98 non-null     object
 5   date         98 non-null     object
 6   textpub      98 non-null     object
dtypes: object(7)
memory usage: 5.5+ KB


In [62]:
file_name = 'habr+20.csv'
df2.to_csv(file_name)